### NLP Assignment-2 U24AI038

You are given a dataset in JSON format text
_
segmentation
_
dataset.json. The dataset is a
snapshot from the large Brown corpus. You need to implement two text segmentation
techniques to segment the text into words and report their performance.
1. Greedy Based Approach that matches the longest word
2. Dynamic Programming Approach that increases the log probability of the text [Hint:
Frequencies of Words are given]
You need to report two evaluation metrics.
1. Accuracy
2. Edit Distance

In [22]:
import json
with open("./Lab-2/text_segmentation_dataset.json", "r") as f:
    data = json.load(f)
    print(data.keys())


dict_keys(['metadata', 'word_counts', 'test_cases'])


In [23]:
metadata = data["metadata"]
test_cases = data["test_cases"]
word_counts = data["word_counts"]

In [24]:
metadata

{'vocabulary_size': 1500,
 'test_case_count': 1000,
 'total_corpus_words': 735040}

#### Greedy based approach that matches the longest word

In [25]:
def edit_distance(incorrect, correct):
    n1 = len(incorrect)
    n2 = len(correct)
    dp = [[float('inf') for i in range(n2+1)] for _ in range(n1+1)]
    for i in range(n1+1):
        dp[i][0] = i
    for i in range(n2+1):
        dp[0][i] =i

    for i in range(1,1+n1):
        for j in range(1,1+n2):
            if incorrect[i-1] == correct[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j] , dp[i][j-1], dp[i-1][j-1])


    return dp[n1][n2]

    
    

In [26]:
edit_distance("Hell", "Hello")

1

#### Logic: Match the longest string before splitting

In [27]:
def longest_match(sentence, words):
    ans = []
    st =0
    while st < len(sentence):
        i = st
        longest_match = st
        while i< len(sentence):
            curr_word = sentence[st:i+1]
            if "".join(curr_word) in words:
                longest_match = i
            i+=1
        ans.append( "".join(sentence[st:longest_match+1] ))
        st = longest_match+1

    return " ".join(ans)
    


    

In [28]:
d = {
    "Hello":0,
    "World":1
}

print(longest_match("HelloWorld", d))

Hello World


In [29]:
longest_match(test_cases[0]['input'], word_counts)

'it that the city takes t e p s to this problem'

In [30]:
output = []
errors = 0 # complete mismatch errors
average_edit_distance_greedy =0
for i in range(len(test_cases)):
    output.append(longest_match(test_cases[i]['input'], word_counts))
    if output[-1] != test_cases[i]['ground_truth']:
        errors+=1
    average_edit_distance_greedy += edit_distance(output[-1], test_cases[i]['ground_truth'])

total = metadata['test_case_count']
accuracy_greedy = ((total-errors)/total) *100
average_edit_distance_greedy /= total
print(f"accuracy is {accuracy_greedy}%" )
print(f"Average edit distance is : {average_edit_distance_greedy} ")
    




accuracy is 69.1%
Average edit distance is : 1.29 


#### DP Approach

In [35]:
# Calculating log probabilities of each word
import math
total_words = metadata['total_corpus_words']
probability = {}
for word in word_counts:
    probability[word] = math.log10(word_counts[word]/total_words)


### Logic evaluate all partitions for a string ending at i and select those partitions which maximize the log probability sum

In [36]:
def dp_match(sentence, probability):
    n = len(sentence)
    dp = [-float('inf') for _ in range(n+1)]
    prev = [-1] * (n+1)
    dp[0] =0

    for i in range(1,n+1):
        for j in range(i):
            curr_word = sentence[j:i]
            if curr_word in probability:
                score = dp[j] + probability[curr_word]
                if score > dp[i]:
                    dp[i] = score
                    prev[i] = j

    i = n
    ans = []
    while i>0:
        j = prev[i]
        if j == -1:
            return "Segmentation Failed"
        ans.append(sentence[j:i])
        i = j

    ans.reverse()
    return " ".join(ans)
        

In [39]:
output = []
errors = 0 # complete mismatch errors
average_edit_distance_dp =0
for i in range(len(test_cases)):
    output.append(dp_match(test_cases[i]['input'], probability))
    if output[-1] != test_cases[i]['ground_truth']:
        errors+=1
    average_edit_distance_dp += edit_distance(output[-1], test_cases[i]['ground_truth'])

total = metadata['test_case_count']
accuracy_dp = ((total-errors)/total) *100
average_edit_distance_dp /= total
print(f"accuracy is {accuracy_dp}%" )
print(f"Average edit distance is : {average_edit_distance_dp} ")
    




accuracy is 98.2%
Average edit distance is : 0.028 


### Summary of results obtained:
>- Greedy Matching
Accuracy is 69.1%
Average edit distance is : 1.29
>- DP Matching
accuracy is 98.2%
Average edit distance is : 0.028